# Step 13b — MJO Preprocessing (lat-aware, 16 × 180)
**Project:** ENSO-BSISO Self-Supervised Learning — MJO Extension  
**Author:** Jiayi (jh9141@nyu.edu)

Variant of `nb13` that **preserves the 15°S–15°N latitude axis** (16 grid points at 2°) instead of meridionally averaging. Output tensor shape is `(N, 3, 16, 180)`, ready for the lat-aware CNN in `nb14b` and `nb15b`.

Why: Session 24 showed the meridionally-averaged input produced a severe seasonal confound in SSL (month-ANOVA F = 300.84). Keeping the lat axis lets the encoder learn N–S structure (Rossby gyres, ITCZ asymmetry) that meridional averaging discards.

Pipeline (same Wheeler & Hendon 2004 simplifications as nb13, just applied per `(lat, lon)`):

| Step | Method | Applied over |
|------|--------|-------------|
| 1. Lat-band subset | keep 15°S–15°N (16 points), **no average** | latitude axis preserved |
| 2. Annual cycle | subtract mean + 3-harmonic Fourier (base **1979–2001**) | per `(lat, lon)` grid point |
| 3. Interannual variability | preceding 120-day running mean | per `(lat, lon)` grid point |
| 4. Global variance normalization | one scalar per variable | std over base × 16 lat × 180 lon |
| 5. Channel order | `[u850, OLR, u200]` | unchanged |
| 6. Output shape | `(N, 3, 16, 180)` | full 2D map per day |

**Inputs** (from `MJO/data/raw/`, reused from nb12):
- `u850_u200_YYYY.nc` × 45 (annual)
- `OLR_MJO_YYYY.nc` × 45
- `rmm_labels.csv`

**Outputs** (to `MJO/lat16/data/processed/`):
- `X_MJO_lat16.npy` shape `(N, 3, 16, 180)` float32 (≈ 567 MB)
- `labels_aligned_mjo_lat16.csv` N rows
- `norm_stats_mjo_lat16.json` global std per channel + metadata
- `latitudes_mjo.npy` (16,) for downstream plotting
- `longitudes_mjo.npy` (180,)

---

## Cell 1 — Mount Google Drive + Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import gc
import numpy as np
import pandas as pd
import xarray as xr

PROJECT_DIR   = '/content/drive/MyDrive/BSISO_SSL_Project'
MJO_DIR       = f'{PROJECT_DIR}/MJO'
RAW_DIR       = f'{MJO_DIR}/data/raw'              # reuse the original raw download
LAT16_DIR     = f'{MJO_DIR}/lat16'                  # NEW root for lat-aware pipeline
PROCESSED_DIR = f'{LAT16_DIR}/data/processed'

os.makedirs(PROCESSED_DIR, exist_ok=True)

print('Google Drive mounted.')
print(f'Raw input  : {RAW_DIR}')
print(f'Output dir : {PROCESSED_DIR}')
print()
print('Raw files (first 10):')
for f in sorted(os.listdir(RAW_DIR))[:10]:
    mb = os.path.getsize(f'{RAW_DIR}/{f}') / 1e6
    print(f'  {f}  ({mb:.1f} MB)')

## Cell 2 — Load ERA5 Wind (u850 + u200, annual chunks)

In [ ]:
wind_files = sorted([
    f'{RAW_DIR}/{f}' for f in os.listdir(RAW_DIR)
    if f.startswith('u850_u200') and f.endswith('.nc')
])

print(f'Found {len(wind_files)} wind files (expected 45 annual files for 1979-2023):')
for f in wind_files[:3]:
    print(f'  {os.path.basename(f)}')
if len(wind_files) > 3:
    print(f'  ... and {len(wind_files)-3} more')

datasets = [xr.open_dataset(f) for f in wind_files]
ds_wind  = xr.concat(datasets, dim='valid_time').sortby('valid_time')

lats_full = ds_wind.latitude.values
lons      = ds_wind.longitude.values
wind_times = pd.DatetimeIndex(ds_wind.valid_time.values).normalize()

print(f'\nCombined wind: {len(wind_times)} days')
print(f'Date range:    {wind_times[0].date()} to {wind_times[-1].date()}')
print(f'Full grid:     {len(lats_full)} lat × {len(lons)} lon')
print(f'Pressure lvls: {sorted(ds_wind.pressure_level.values.tolist())} hPa')
print(f'Lat range:     {lats_full.min():.1f} to {lats_full.max():.1f}')
print(f'Lon range:     {lons.min():.1f} to {lons.max():.1f}')

## Cell 3 — Load ERA5 OLR

In [ ]:
olr_files = sorted([
    f'{RAW_DIR}/{f}' for f in os.listdir(RAW_DIR)
    if f.startswith('OLR_MJO_') and f.endswith('.nc')
])
print(f'Found {len(olr_files)} OLR files (expected 45)')

ds_olr    = xr.concat([xr.open_dataset(f) for f in olr_files],
                       dim='valid_time').sortby('valid_time')
olr_times = pd.DatetimeIndex(ds_olr.valid_time.values).normalize()

print(f'OLR combined: {len(olr_times)} days')
print(f'Date range:   {olr_times[0].date()} to {olr_times[-1].date()}')
print(f'Grid:         {len(ds_olr.latitude)} lat × {len(ds_olr.longitude)} lon')
print(f'Raw ttr range: [{float(ds_olr["ttr"].min()):.0f}, {float(ds_olr["ttr"].max()):.0f}] J/m²')

assert len(ds_olr.latitude) == len(lats_full), 'OLR/wind lat mismatch'
assert len(ds_olr.longitude) == len(lons),      'OLR/wind lon mismatch'

## Cell 4 — Load RMM Labels

In [ ]:
df_labels = pd.read_csv(f'{RAW_DIR}/rmm_labels.csv', parse_dates=['date'])
df_labels['date'] = df_labels['date'].dt.normalize()

print(f'Labels: {len(df_labels)} rows')
print(f'Date range: {df_labels["date"].min().date()} to {df_labels["date"].max().date()}')
print()
print('ENSO distribution:')
print(df_labels['enso_category'].value_counts())
print()
active = df_labels[~df_labels['weak_mjo']]
print(f'Active MJO days (ampl >= 1.0, phase 1-8): {len(active)}')
print(active['phase'].value_counts().sort_index())

## Cell 5 — Step 1: Subset to 15°S–15°N Lat Band  *(NO meridional average)*

Keep all latitude grid points inside 15°S–15°N (16 points at 2°). Resulting arrays are 3D `(T, 16, 180)` per variable — the latitude axis is preserved end-to-end, in contrast with nb13 which collapsed it here.

In [ ]:
common_dates = wind_times.intersection(olr_times).sort_values()
T = len(common_dates)
print(f'Common dates: {T}  (expected ~16,425 for 45 years all-month)')

wind_date_to_idx = {d: i for i, d in enumerate(wind_times)}
olr_date_to_idx  = {d: i for i, d in enumerate(olr_times)}
wind_idx = np.array([wind_date_to_idx[d] for d in common_dates])
olr_idx  = np.array([olr_date_to_idx[d]  for d in common_dates])

# Pick 15S-15N latitude band (inclusive). At 2° this should give 16 points.
lat_mask = (lats_full >= -15.0) & (lats_full <= 15.0)
lats = lats_full[lat_mask]
n_lat, n_lon = len(lats), len(lons)
print(f'\nLatitudes kept ({n_lat} pts): {lats}')
assert n_lat == 16, f'Expected 16 lat points at 2° in 15S-15N, got {n_lat}'

print('\nLoading u850 ...')
u850 = ds_wind['u'].sel(pressure_level=850).values[wind_idx][:, lat_mask, :].astype(np.float32)
print('Loading u200 ...')
u200 = ds_wind['u'].sel(pressure_level=200).values[wind_idx][:, lat_mask, :].astype(np.float32)
print('Loading OLR ...')
olr  = (-ds_olr['ttr'].values[olr_idx][:, lat_mask, :]).astype(np.float32)

print(f'\nShapes (T, n_lat, n_lon) per variable:')
print(f'  u850: {u850.shape}')
print(f'  u200: {u200.shape}')
print(f'  OLR:  {olr.shape}')
print(f'Memory per variable: {u850.nbytes / 1e6:.1f} MB × 3 = {3*u850.nbytes/1e6:.0f} MB')

# Free xarray datasets and lists
del datasets, ds_wind, ds_olr
gc.collect()

## Cell 6 — Step 2: Remove Annual Cycle (3 Fourier Harmonics, base 1979–2001)

Same 3-harmonic Fourier fit as nb13 but applied **per `(lat, lon)` grid point** — implemented by reshaping the spatial dims into one flat axis, running the existing per-column lstsq, and reshaping back.

In [ ]:
BASE_START  = 1979
BASE_END    = 2001
N_HARMONICS = 3
PERIOD      = 365

years     = common_dates.year.values
doys      = common_dates.day_of_year.values
base_mask = (years >= BASE_START) & (years <= BASE_END)

print(f'Base period: {BASE_START}-{BASE_END}')
print(f'Base period days available: {base_mask.sum()}')

def build_fourier_features(d, K=3, P=365):
    feats = [np.ones(len(d))]
    for k in range(1, K + 1):
        feats.append(np.cos(2 * np.pi * k * d / P))
        feats.append(np.sin(2 * np.pi * k * d / P))
    return np.column_stack(feats)

def remove_annual_cycle_harmonic_3d(field_3d, doys, base_mask, K=3, P=365):
    """
    field_3d: (T, n_lat, n_lon) float32
    Returns: (T, n_lat, n_lon) float32 anomaly after subtracting smooth annual cycle.
    Implementation: flatten (lat, lon) -> single col axis, run per-column lstsq, reshape.
    """
    T, nlat, nlon = field_3d.shape
    field_flat = field_3d.reshape(T, nlat * nlon)

    unique_doys = np.unique(doys)
    n_doys = len(unique_doys)

    # Step a: DOY climatology over base period -> (n_doys, n_lat * n_lon)
    clim = np.zeros((n_doys, nlat * nlon), dtype=np.float64)
    for i, d in enumerate(unique_doys):
        ids = np.where((doys == d) & base_mask)[0]
        if len(ids) > 0:
            clim[i] = field_flat[ids].mean(axis=0)

    # Step b: fit K harmonics at every grid point
    X_fit  = build_fourier_features(unique_doys, K, P)
    coeffs, _, _, _ = np.linalg.lstsq(X_fit, clim, rcond=None)

    # Step c: evaluate smooth cycle for every observed day and subtract
    X_all  = build_fourier_features(doys, K, P)
    smooth = (X_all @ coeffs).astype(np.float32)
    return (field_flat - smooth).reshape(T, nlat, nlon)

print(f'\nRemoving {N_HARMONICS}-harmonic annual cycle from u850 (per grid point) ...')
u850_a = remove_annual_cycle_harmonic_3d(u850, doys, base_mask, N_HARMONICS, PERIOD)
print(f'Removing {N_HARMONICS}-harmonic annual cycle from u200 (per grid point) ...')
u200_a = remove_annual_cycle_harmonic_3d(u200, doys, base_mask, N_HARMONICS, PERIOD)
print(f'Removing {N_HARMONICS}-harmonic annual cycle from OLR  (per grid point) ...')
olr_a  = remove_annual_cycle_harmonic_3d(olr,  doys, base_mask, N_HARMONICS, PERIOD)

del u850, u200, olr
gc.collect()

print(f'\nPost-step-2 mean over base period (should be ~0):')
print(f'  u850: {u850_a[base_mask].mean():+.6f}')
print(f'  u200: {u200_a[base_mask].mean():+.6f}')
print(f'  OLR:  {olr_a[base_mask].mean():+.6f}')
print(f'\nPost-step-2 std over base period:')
print(f'  u850: {u850_a[base_mask].std():.4f} m/s')
print(f'  u200: {u200_a[base_mask].std():.4f} m/s')
print(f'  OLR:  {olr_a[base_mask].std():.2f} J/m²')

## Cell 7 — Step 3: Remove Interannual Variability (120-day Running Mean per Grid Point)

Subtract preceding 120-day running mean at every `(lat, lon)` grid point. Same Session-21 NaN guard as nb13 (first day has no preceding window; fall back to step-2 anomaly).

Implementation: flatten `(lat, lon)` into the column axis, use pandas rolling on the resulting 2D DataFrame (vectorized over all 16 × 180 = 2880 columns), reshape back.

In [ ]:
WINDOW = 120

def remove_running_mean_3d(anom_3d, dates, window=120):
    """
    Subtract preceding `window`-day running mean per (lat, lon) grid point.
    anom_3d: (T, n_lat, n_lon) float32, dates: DatetimeIndex of length T.
    closed='left' excludes the current day (strictly preceding window).
    min_periods=1: requires >=1 preceding observation; day 0 has 0 -> returns NaN.
    Day-0 NaNs are replaced with the unchanged step-2 anomaly (no correction).
    """
    T, nlat, nlon = anom_3d.shape
    flat = anom_3d.reshape(T, nlat * nlon)
    df = pd.DataFrame(flat.astype(np.float64), index=dates)
    rm = df.rolling(window=window, min_periods=1, closed='left').mean().values
    result = (flat - rm).astype(np.float32)
    nan_days = np.where(np.isnan(result).any(axis=1))[0]
    if len(nan_days) > 0:
        print(f'  Patched {len(nan_days)} NaN day(s) at indices {nan_days.tolist()} with raw anomaly')
        result[nan_days] = flat[nan_days]
    return result.reshape(T, nlat, nlon)

print('Removing 120-day preceding running mean from u850 ...')
u850_iso = remove_running_mean_3d(u850_a, common_dates, WINDOW)
print('Removing 120-day preceding running mean from u200 ...')
u200_iso = remove_running_mean_3d(u200_a, common_dates, WINDOW)
print('Removing 120-day preceding running mean from OLR  ...')
olr_iso  = remove_running_mean_3d(olr_a,  common_dates, WINDOW)

del u850_a, u200_a, olr_a
gc.collect()

for name, arr in [('u850', u850_iso), ('u200', u200_iso), ('OLR', olr_iso)]:
    n_nan = int(np.isnan(arr).sum())
    print(f'  {name}: {n_nan} NaN values after running mean  (expected 0)')

print(f'\nPreceding-days window coverage (sanity check):')
for i in [0, 1, 10, 119, 120]:
    avail = min(i, WINDOW)
    print(f'  Day {i:3d} ({str(common_dates[i].date())}): {avail}/{WINDOW} preceding days')

## Cell 8 — Step 4: Global Variance Normalization (one scalar per variable)

Compute a single std per variable over (base-period days) × (16 lat) × (180 lon), then divide. Preserves WH04's “each variable contributes equally to the combined representation” intent over the full 2D field.

In [ ]:
def global_std_normalize_3d(iso_3d, base_mask):
    base_vals  = iso_3d[base_mask].ravel()
    scalar_std = float(np.nanstd(base_vals))
    return (iso_3d / scalar_std).astype(np.float32), scalar_std

u850_n, std_u850 = global_std_normalize_3d(u850_iso, base_mask)
u200_n, std_u200 = global_std_normalize_3d(u200_iso, base_mask)
olr_n,  std_olr  = global_std_normalize_3d(olr_iso,  base_mask)

del u850_iso, u200_iso, olr_iso
gc.collect()

norm_stats = {
    'method'              : 'global_temporal_std over (base_period, lat, lon)  [WH04, lat-aware]',
    'base_period'         : f'{BASE_START}-{BASE_END}',
    'n_harmonics'         : N_HARMONICS,
    'running_mean_window' : WINDOW,
    'meridional_band'     : '15S-15N (16 grid points, axis preserved)',
    'enso_removal'        : 'simplified Lee (120-day running mean only, no SST1)',
    'channel_order'       : ['u850', 'OLR', 'u200'],
    'tensor_shape'        : '(N, 3, 16, 180)',
    'u850' : {'global_std': std_u850},
    'OLR'  : {'global_std': std_olr},
    'u200' : {'global_std': std_u200},
}

print('Global temporal std (base period, over lat×lon×days):')
print(f'  u850: {std_u850:.4f} m/s')
print(f'  u200: {std_u200:.4f} m/s')
print(f'  OLR:  {std_olr:.4f} J/m²')

print(f'\nPost-normalization std over base period (should be ~1):')
print(f'  u850: {np.nanstd(u850_n[base_mask]):.4f}')
print(f'  u200: {np.nanstd(u200_n[base_mask]):.4f}')
print(f'  OLR:  {np.nanstd(olr_n[base_mask]):.4f}')

print(f'\nNaN check (should all be 0):')
for name, arr in [('u850_n', u850_n), ('u200_n', u200_n), ('olr_n', olr_n)]:
    print(f'  {name}: {int(np.isnan(arr).sum())} NaN')

## Cell 9 — Stack Channels & Align with Labels

Channel order `[u850, OLR, u200]` (same as nb13). Final tensor shape `(N, 3, 16, 180)` — no singleton lat axis.

In [ ]:
label_dates = pd.DatetimeIndex(df_labels['date'])
aligned     = common_dates.intersection(label_dates).sort_values()

print(f'ERA5 dates:      {len(common_dates)}')
print(f'Label dates:     {len(label_dates)}')
print(f'Aligned (both):  {len(aligned)}')
print(f'Date range:      {aligned[0].date()} to {aligned[-1].date()}')

common_date_to_idx = {d: i for i, d in enumerate(common_dates)}
sel_idx = np.array([common_date_to_idx[d] for d in aligned])

X_MJO_lat16 = np.stack([
    u850_n[sel_idx],
    olr_n[sel_idx],
    u200_n[sel_idx],
], axis=1).astype(np.float32)

print(f'\nX_MJO_lat16 shape: {X_MJO_lat16.shape}  (expect (~16425, 3, 16, 180))')
print(f'Memory:            {X_MJO_lat16.nbytes / 1e6:.1f} MB')

del u850_n, u200_n, olr_n
gc.collect()

df_aligned = df_labels[df_labels['date'].isin(aligned)].copy()
df_aligned = df_aligned.sort_values('date').reset_index(drop=True)

assert list(df_aligned['date']) == list(aligned), 'Date order mismatch!'
print(f'\nLabels aligned: {len(df_aligned)} rows')
print(df_aligned[['date', 'rmm1', 'rmm2', 'phase', 'amplitude', 'enso_category', 'weak_mjo']].head(5))

## Cell 10 — Save Outputs

In [ ]:
X_path     = f'{PROCESSED_DIR}/X_MJO_lat16.npy'
y_path     = f'{PROCESSED_DIR}/labels_aligned_mjo_lat16.csv'
stats_path = f'{PROCESSED_DIR}/norm_stats_mjo_lat16.json'
lat_path   = f'{PROCESSED_DIR}/latitudes_mjo.npy'
lon_path   = f'{PROCESSED_DIR}/longitudes_mjo.npy'

np.save(X_path, X_MJO_lat16)
df_aligned.to_csv(y_path, index=False)
np.save(lat_path, lats.astype(np.float32))
np.save(lon_path, lons.astype(np.float32))
with open(stats_path, 'w') as f:
    json.dump(norm_stats, f, indent=2)

print('Saved:')
for p in [X_path, y_path, stats_path, lat_path, lon_path]:
    mb = os.path.getsize(p) / 1e6
    print(f'  {os.path.basename(p):35s}  ({mb:.2f} MB)')

## Cell 11 — Verification: RMM Phase Composite Maps (lat × lon)

8-panel grid (one per RMM phase) of mean OLR' over active-MJO days. Expected MJO signature: convection (negative OLR') propagates eastward through phases 1 → 8. Now with lat dimension visible, we can additionally inspect:
- Off-equator structure (e.g., Rossby gyres slightly N and S of the equator)
- N–S asymmetries (ITCZ shifts, seasonal hemispheric bias)

These were averaged away in nb13.

In [ ]:
import matplotlib.pyplot as plt

active = df_aligned[~df_aligned['weak_mjo'] & (df_aligned['phase'].between(1, 8))]
olr_chan = 1   # channel index for OLR

fig, axes = plt.subplots(4, 2, figsize=(14, 10), sharex=True, sharey=True)
fig.suptitle('MJO Phase Composites — OLR\' anomaly (σ)  [lat×lon, 15°S–15°N]',
             fontsize=13, fontweight='bold')

# Pre-compute global vmax so all panels share the same color scale.
all_phase_means = []
for ph in range(1, 9):
    idx = active[active['phase'] == ph].index.values
    if len(idx) > 0:
        all_phase_means.append(X_MJO_lat16[idx, olr_chan].mean(axis=0))
vmax = float(np.percentile(np.abs(np.array(all_phase_means)), 99))

for ax, ph in zip(axes.flat, range(1, 9)):
    idx = active[active['phase'] == ph].index.values
    if len(idx) == 0:
        ax.set_title(f'Phase {ph} — no data')
        continue
    olr_map = X_MJO_lat16[idx, olr_chan].mean(axis=0)   # (16, 180)
    im = ax.imshow(olr_map, cmap='RdBu_r', aspect='auto',
                   extent=[lons.min(), lons.max(), lats.min(), lats.max()],
                   vmin=-vmax, vmax=vmax, origin='lower')
    ax.set_title(f'Phase {ph}  (N={len(idx)})', fontsize=10)
    ax.axhline(0, color='k', lw=0.5, alpha=0.5)

for ax in axes[-1]:
    ax.set_xlabel('Longitude (°)')
for ax in axes[:, 0]:
    ax.set_ylabel('Latitude (°)')

fig.subplots_adjust(right=0.92)
cbar_ax = fig.add_axes([0.94, 0.15, 0.015, 0.7])
plt.colorbar(im, cax=cbar_ax, label="OLR' (σ)")
fig_path = f'{PROCESSED_DIR}/mjo_phase_composites_lat16.png'
plt.savefig(fig_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')
print('Look for the negative-OLR (blue) patch shifting eastward as phase increases, plus any N–S asymmetry.')

## Cell 12 — Verification: Zonal Hovmöller Diagram (1992–93 strong MJO)

Hovmöller plots are by convention meridionally averaged. We display the average here only — the saved tensor itself still has the full lat axis. Eastward propagation appears as bands tilted from upper-left to lower-right.

In [ ]:
HOV_START = pd.Timestamp('1992-11-01')
HOV_END   = pd.Timestamp('1993-03-31')

date_arr = pd.DatetimeIndex(df_aligned['date'])
hov_mask = (date_arr >= HOV_START) & (date_arr <= HOV_END)
hov_idx  = np.where(hov_mask)[0]

# Average over lat for display
olr_hov  = X_MJO_lat16[hov_idx, 1].mean(axis=1)   # (T_hov, 180)
u850_hov = X_MJO_lat16[hov_idx, 0].mean(axis=1)
hov_dates = date_arr[hov_idx]

print(f'Hovmöller window: {hov_dates[0].date()} to {hov_dates[-1].date()}  ({len(hov_idx)} days)')

fig, axes = plt.subplots(1, 2, figsize=(15, 8))
fig.suptitle(f'Zonal Hovmöller (lat-averaged for display)  |  {HOV_START.date()} to {HOV_END.date()}',
             fontsize=12, fontweight='bold')

for ax, data, title in zip(
    axes,
    [olr_hov, u850_hov],
    ["OLR' anomaly (negative = convection)", "u850' anomaly"],
):
    vmax = float(np.percentile(np.abs(data), 95))
    im = ax.imshow(data, cmap='RdBu_r', aspect='auto',
                   extent=[lons.min(), lons.max(), len(hov_idx), 0],
                   vmin=-vmax, vmax=vmax)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Longitude (°)')
    ax.set_ylabel('Days from start')
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.04, label='σ')

plt.tight_layout()
fig_path = f'{PROCESSED_DIR}/mjo_hovmoller_1992_93_lat16.png'
plt.savefig(fig_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

## Cell 13 — Verification: ENSO Composite Maps (should be ≈ 0)

After Lee preprocessing the residual ENSO signal in the input fields should be small. We display 2D maps now (instead of nb13's 1D longitude profiles) to confirm there's no leftover N–S structure either.

In [ ]:
enso_cats = ['El Nino', 'Neutral', 'La Nina']
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True, sharey=True)
fig.suptitle('ENSO composites (OLR\' anomaly map) — should be near-zero after Lee preprocessing',
             fontsize=12, fontweight='bold')

olr_all_means = []
for cat in enso_cats:
    idx = df_aligned[df_aligned['enso_category'] == cat].index.values
    if len(idx) > 0:
        olr_all_means.append(X_MJO_lat16[idx, 1].mean(axis=0))
vmax = float(np.percentile(np.abs(np.array(olr_all_means)), 99))

for ax, cat in zip(axes, enso_cats):
    idx = df_aligned[df_aligned['enso_category'] == cat].index.values
    if len(idx) == 0:
        ax.set_title(f'{cat} — no data')
        continue
    olr_map = X_MJO_lat16[idx, 1].mean(axis=0)
    im = ax.imshow(olr_map, cmap='RdBu_r', aspect='auto',
                   extent=[lons.min(), lons.max(), lats.min(), lats.max()],
                   vmin=-vmax, vmax=vmax, origin='lower')
    ax.set_title(f'{cat}  (N={len(idx)})   max|map| = {np.abs(olr_map).max():.3f}σ', fontsize=10)
    ax.axhline(0, color='k', lw=0.5, alpha=0.5)
    ax.set_ylabel('Latitude (°)')

axes[-1].set_xlabel('Longitude (°)')
fig.subplots_adjust(right=0.92)
cbar_ax = fig.add_axes([0.94, 0.15, 0.015, 0.7])
plt.colorbar(im, cax=cbar_ax, label="OLR' (σ)")
fig_path = f'{PROCESSED_DIR}/mjo_enso_composites_lat16.png'
plt.savefig(fig_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')
print('Interpretation: small |map| (≪ 0.5σ) -> background removed; large -> leftover ENSO signal.')

## Cell 14 — Summary Report

In [ ]:
print('=' * 65)
print('MJO PREPROCESSING SUMMARY  (lat-aware, 16 × 180)')
print('=' * 65)
print(f'Method:               Wheeler & Hendon (2004), simplified, lat-axis preserved')
print(f'Input data:           ERA5 all-year, 1979-2023, 15°S–15°N strip')
print(f'Base period:          {BASE_START}-{BASE_END}')
print(f'Annual cycle:         {N_HARMONICS}-harmonic Fourier (per lat × lon grid point)')
print(f'ENSO removal:         {WINDOW}-day preceding running mean (no SST1)')
print(f'Normalization:        global temporal std over (base, lat, lon)')
print()
print(f'Output array:         X_MJO_lat16.npy  shape={X_MJO_lat16.shape}  float32')
print(f'Channel order:        [u850, OLR, u200]')
print(f'Labels:               labels_aligned_mjo_lat16.csv  {len(df_aligned)} rows')
print()
print('Normalization scalars (global temporal std, base period):')
print(f'  u850: {std_u850:.4f} m/s')
print(f'  u200: {std_u200:.4f} m/s')
print(f'  OLR:  {std_olr:.4f} J/m²')
print()
print('Label coverage:')
print(f'  Total days:                {len(df_aligned)}')
print(f'  Active MJO (ampl >= 1.0):  {(~df_aligned["weak_mjo"]).sum()}')
print(f'  ENSO  El Nino / Neutral / La Nina:  '
      f'{(df_aligned["enso_category"]=="El Nino").sum()} / '
      f'{(df_aligned["enso_category"]=="Neutral").sum()} / '
      f'{(df_aligned["enso_category"]=="La Nina").sum()}')
print()
print('Next: nb14b (supervised 2D, lat-aware) and nb15b (SSL temporal 2D, lat-aware)')
print('=' * 65)

---
## Done!

Google Drive should now contain:

```
BSISO_SSL_Project/MJO/lat16/data/processed/
├── X_MJO_lat16.npy                    (N, 3, 16, 180) float32
├── labels_aligned_mjo_lat16.csv       N rows
├── norm_stats_mjo_lat16.json          metadata + global stds
├── latitudes_mjo.npy                  (16,) for plotting
├── longitudes_mjo.npy                 (180,)
├── mjo_phase_composites_lat16.png
├── mjo_hovmoller_1992_93_lat16.png
└── mjo_enso_composites_lat16.png
```

**Verification checklist before moving on:**

1. Phase composite map: convection patch (blue / negative OLR') shifts eastward 1 → 8
2. Hovmöller: visible eastward-tilted band of negative OLR' during 1992–93
3. ENSO composite map: max\|composite\| < 0.5σ for all three categories (Lee removed the background)
4. Shape: `X_MJO_lat16.shape == (N, 3, 16, 180)` with N ≈ 16,425
5. No NaN in the final array

**Next:**
- **nb14b** (`14b_mjo_supervised_2d_lat16.ipynb`) — supervised CNN with RMM phase + ENSO labels, lat-aware architecture
- **nb15b** (`15b_mjo_ssl_temporal_2d_lat16.ipynb`) — SSL temporal encoder with 20–90 day bandpass, lat-aware architecture

---
*DDCS Project | jh9141@nyu.edu*